# nb_00 — (instructor only) regenerate the source data — lab

You normally **don't run this**. The lab consumes pre-generated files hosted in
the repo's `/data` folder. This reproduces them deterministically if you want to
change volumes or reshape the scenario. Run it in a plain Python kernel; it writes
a local `./data` folder you then commit to your fork.

> **Lab notebook.** The code cell below only sketches the generator's structure as
> `# TODO` comments — see the solution notebook of the same name for reference on the
> exact shapes each output file needs (columns, ranges, and design hooks like
> replayed events, quarantine cases, and SCD2 history). This is optional and only
> useful if you want to author your own dataset variant.


In [ ]:
# TODO: Generate the lab's synthetic source data, deterministically (fix a random seed).
#
# The generator must produce, under ./data:
#   reference/workers.csv              initial worker snapshot   (SCD2 load 1)
#   reference/workers_delta.csv        later worker snapshot     (SCD2 load 2)
#   reference/cost_centers.csv         cost-centre master        (SCD1, carries RLS region)
#   feeds/pay_bands_feed.json          nested pay-grid history   (SCD2 from history)
#   feeds/fx_rates.csv                 monthly FX -> CAD
#   events/workforce_events_YYYY-MM.csv  60 monthly HR event extracts (pipeline ForEach)
#
# Design hooks the rest of the lab depends on:
#   - dedup:      replay a small percentage of events with the same event_id but a later
#                 ingest_ts, so nb_02 has real duplicates to remove
#   - quarantine: sprinkle in negative amounts, orphan employee_id values, bad dates, and
#                 unknown currencies, so nb_02's data-quality rules have real failures to catch
#   - conform:    have one "source system" emit ISO-2 work-country codes and another emit
#                 ISO-3, so nb_02 has a real conforming step to write
#   - SCD2:       re-benchmark the pay grids annually (feeds/pay_bands_feed.json) and change
#                 some worker attributes between the two worker snapshots, so nb_03a has real
#                 history to turn into SCD2 dimensions
#
# Suggested steps:
#   1. Recreate (and empty) the reference/, feeds/, and events/ output folders.
#   2. Build the pay-grid history per (classification group, level, year), applying an
#      annual economic increase, and write it as nested JSON (one object per group/level with
#      a band_history array).
#   3. Build monthly FX rates per currency, anchored near a realistic rate and drifting
#      randomly month to month; CAD is always 1.0.
#   4. Build the cost-centre master (id, name, branch, hr_region, business_line).
#   5. Build the initial worker snapshot and a later "delta" snapshot with some workers
#      changed (classification/directorate/employment type) and some brand new.
#   6. Build monthly workforce-event extracts (hires, promotions, step increments,
#      performance pay, deployments, leaves, departures) with the dedup/quarantine hooks
#      above baked in, one CSV per month.
#   7. Run the generators and write every file; then run a lightweight validation pass that
#      sanity-checks row counts and referential integrity before you commit ./data.
